# Imports

In [17]:
from pathlib import Path
print(Path.cwd())

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent)) # problem with dependency resolution (e.g. custom_builder) without this

/Users/mac/Documents/dev/ID2221/dic/Week 2


In [18]:
# Use delta features if needed (DeltaTable, etc.)
from delta import *
from custom_builder import builder
from log import *
import numpy as np
from Queries import *

# use the existing preconfigured builder to create the Spark session.
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f'current database: {spark.catalog.currentDatabase()}')
print(f'spark tables: {spark.catalog.listTables()}')

from pyspark.sql import functions as F

import json

current database: default
spark tables: [Table(name='air_quality', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='integrated_taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_trips', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='taxi_zone_lookup', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False), Table(name='weather', catalog='spark_catalog', namespace=['default'], description=None, tableType='MANAGED', isTemporary=False)]


26/09/16 18:28:08 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


# Ensure uncached tables?

In [19]:
spark.catalog.uncacheTable("default.air_quality")
spark.catalog.uncacheTable("default.taxi_trips")
spark.catalog.uncacheTable("default.taxi_zone_lookup")
spark.catalog.uncacheTable("default.weather")

# Ensure no auto broadcast?

In [20]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

# reset the broadcast threshold to default value
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)  # 10 MB

# Disable AQE

In [21]:
# disable AQE
spark.conf.set("spark.sql.adaptive.enabled", "false")

## Queries

### Query 2.1

In [37]:
result = spark.sql(query_2_1())
result.show()
result.explain(mode="formatted")

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|  Van Cortlandt Park|    1|       18|
|            Kips Bay|    1|    32984|
|                SoHo|    1|    20197|
|Upper West Side S...|    1|    88474|
|Upper East Side S...|    1|   142708|
|           Stapleton|    1|        4|
|       Rockaway Park|    1|       80|
|East New York/Pen...|    1|      245|
|          Bath Beach|    1|       58|
|Upper East Side N...|   12|        1|
|  Claremont/Bathgate|    1|      173|
|Bay Terrace/Fort ...|    1|       38|
|       Fordham South|    1|       91|
|             Bayside|    1|       81|
|    Garment District|    1|    48093|
|     Cambria Heights|    1|      143|
|    Bensonhurst East|    1|      145|
|Governor's Island...|    1|        1|
|         Great Kills|    1|        1|
|Upper West Side N...|    1|    64234|
+--------------------+-----+---------+
only showing top 20 rows
== Physical Plan ==
* HashAggregate (14

### Query 2.2

In [38]:
result = spark.sql(query_2_2())
result.show()
result.explain(mode="formatted")

+--------------+-------+------------------+
|  column_group|    cnt| avg_trip_distance|
+--------------+-------+------------------+
|greater_than_0| 424773| 3.470743355446981|
|  zero_or_null|2539795|3.6824345463125323|
+--------------+-------+------------------+

== Physical Plan ==
* HashAggregate (14)
+- Exchange (13)
   +- * HashAggregate (12)
      +- * Project (11)
         +- * SortMergeJoin LeftOuter (10)
            :- * Sort (4)
            :  +- Exchange (3)
            :     +- * ColumnarToRow (2)
            :        +- Scan parquet spark_catalog.default.taxi_trips (1)
            +- * Sort (9)
               +- Exchange (8)
                  +- * Filter (7)
                     +- * ColumnarToRow (6)
                        +- Scan parquet spark_catalog.default.weather (5)


(1) Scan parquet spark_catalog.default.taxi_trips
Output [2]: [pu_datetime#10227, trip_distance#10232]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/mac/Documents/dev/ID2221/dic/spark_pr

### Query 2.3

In [39]:
res = spark.sql(query_2_3())
res.show()
res.explain(mode="formatted")

+-----------+-------+
|measurement|  trips|
+-----------+-------+
|       NULL|2945398|
|        1.3|     20|
|        1.6|     13|
|        1.7|     31|
|        1.8|     41|
|        1.9|     19|
|        2.0|     24|
|        2.1|    135|
|        2.2|     39|
|        2.3|     36|
|        2.4|     51|
|        2.5|    137|
|        2.6|     93|
|        2.7|     84|
|        2.8|     27|
|        2.9|    141|
|        3.0|     62|
|        3.1|     89|
|        3.2|    115|
|        3.3|     71|
+-----------+-------+
only showing top 20 rows
== Physical Plan ==
* Sort (33)
+- Exchange (32)
   +- * HashAggregate (31)
      +- Exchange (30)
         +- * HashAggregate (29)
            +- * Project (28)
               +- * SortMergeJoin LeftOuter (27)
                  :- * Sort (13)
                  :  +- Exchange (12)
                  :     +- * Project (11)
                  :        +- * SortMergeJoin LeftOuter (10)
                  :           :- * Sort (4)
                  

### Query 2.5

In [40]:
res = spark.sql(query_2_5())
res.show()
res.explain(mode="formatted")

+---+----+-----+
|day|hour|trips|
+---+----+-----+
|Fri|  18|29050|
|Fri|  17|28034|
|Fri|  19|26656|
|Fri|  16|25645|
|Fri|  15|25578|
|Fri|  14|24746|
|Fri|  22|24015|
|Fri|  13|22190|
|Fri|  23|22100|
|Fri|  21|21976|
|Fri|  20|21168|
|Fri|  12|21024|
|Fri|  11|19852|
|Fri|  10|19559|
|Fri|   9|18192|
|Fri|   8|17323|
|Fri|   7|13043|
|Fri|   0| 8804|
|Fri|   6| 6283|
|Fri|   1| 4805|
+---+----+-----+
only showing top 20 rows
== Physical Plan ==
* Sort (8)
+- Exchange (7)
   +- * HashAggregate (6)
      +- Exchange (5)
         +- * HashAggregate (4)
            +- * Project (3)
               +- * ColumnarToRow (2)
                  +- Scan parquet spark_catalog.default.taxi_trips (1)


(1) Scan parquet spark_catalog.default.taxi_trips
Output [1]: [pu_datetime#11054]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/mac/Documents/dev/ID2221/dic/spark_project/spark-warehouse/taxi_trips]
ReadSchema: struct<pu_datetime:timestamp>

(2) ColumnarToRow [codegen id : 1]
Input [1]

### Query 2.6

In [41]:
res = spark.sql(query_2_6())
res.show()
res.explain(mode="formatted")

+-----+-------+
|month|  trips|
+-----+-------+
|  Dec|     12|
|  Feb|      3|
|  Jan|2964553|
+-----+-------+

== Physical Plan ==
* Sort (8)
+- Exchange (7)
   +- * HashAggregate (6)
      +- Exchange (5)
         +- * HashAggregate (4)
            +- * Project (3)
               +- * ColumnarToRow (2)
                  +- Scan parquet spark_catalog.default.taxi_trips (1)


(1) Scan parquet spark_catalog.default.taxi_trips
Output [1]: [pu_datetime#11236]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/mac/Documents/dev/ID2221/dic/spark_project/spark-warehouse/taxi_trips]
ReadSchema: struct<pu_datetime:timestamp>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [pu_datetime#11236]

(3) Project [codegen id : 1]
Output [1]: [date_format(pu_datetime#11236, MMM, Some(Europe/Stockholm)) AS _groupingexpression#11332]
Input [1]: [pu_datetime#11236]

(4) HashAggregate [codegen id : 1]
Input [1]: [_groupingexpression#11332]
Keys [1]: [_groupingexpression#11332]
Functions [1]: [parti

# Enable AQE

In [42]:
# enable AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")

## Queries

### Query 2.1

In [58]:
result = spark.sql(query_2_1())
result.show()
result.explain(mode="formatted")

+--------------------+-----+---------+
|             pu_zone|month|row_count|
+--------------------+-----+---------+
|            Kips Bay|    1|    32984|
|                SoHo|    1|    20197|
|Upper East Side N...|   12|        1|
|Breezy Point/Fort...|    1|        6|
|    Bensonhurst West|    1|      153|
|       East New York|    1|      938|
|Flushing Meadows-...|    1|      510|
|        Battery Park|    1|      875|
|        Borough Park|    1|      237|
| Grymes Hill/Clifton|    1|        2|
|          Kensington|    1|      127|
|         Hunts Point|    1|      105|
|       Melrose South|    1|      289|
|       Prospect Park|    1|       53|
|          Pelham Bay|    1|       47|
| UN/Turtle Bay South|    1|    33595|
|Washington Height...|    1|      495|
|       Willets Point|    1|       14|
|Downtown Brooklyn...|    1|     1393|
|    Inwood Hill Park|    1|       22|
+--------------------+-----+---------+
only showing top 20 rows
== Physical Plan ==
AdaptiveSparkPlan (

### Query 2.2

In [54]:
result = spark.sql(query_2_2())
result.show()
result.explain(mode="formatted")

+--------------+-------+------------------+
|  column_group|    cnt| avg_trip_distance|
+--------------+-------+------------------+
|greater_than_0| 424773| 3.470743355446981|
|  zero_or_null|2539795|3.6824345463125323|
+--------------+-------+------------------+

== Physical Plan ==
AdaptiveSparkPlan (13)
+- HashAggregate (12)
   +- Exchange (11)
      +- HashAggregate (10)
         +- Project (9)
            +- SortMergeJoin LeftOuter (8)
               :- Sort (3)
               :  +- Exchange (2)
               :     +- Scan parquet spark_catalog.default.taxi_trips (1)
               +- Sort (7)
                  +- Exchange (6)
                     +- Filter (5)
                        +- Scan parquet spark_catalog.default.weather (4)


(1) Scan parquet spark_catalog.default.taxi_trips
Output [2]: [pu_datetime#14769, trip_distance#14774]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/mac/Documents/dev/ID2221/dic/spark_project/spark-warehouse/taxi_trips]
ReadSchema: st

### Query 2.3

In [55]:
res = spark.sql(query_2_3())
res.show()
res.explain(mode="formatted")

+-----------+-------+
|measurement|  trips|
+-----------+-------+
|       NULL|2945398|
|        1.3|     20|
|        1.6|     13|
|        1.7|     31|
|        1.8|     41|
|        1.9|     19|
|        2.0|     24|
|        2.1|    135|
|        2.2|     39|
|        2.3|     36|
|        2.4|     51|
|        2.5|    137|
|        2.6|     93|
|        2.7|     84|
|        2.8|     27|
|        2.9|    141|
|        3.0|     62|
|        3.1|     89|
|        3.2|    115|
|        3.3|     71|
+-----------+-------+
only showing top 20 rows
== Physical Plan ==
AdaptiveSparkPlan (31)
+- Sort (30)
   +- Exchange (29)
      +- HashAggregate (28)
         +- Exchange (27)
            +- HashAggregate (26)
               +- Project (25)
                  +- SortMergeJoin LeftOuter (24)
                     :- Sort (11)
                     :  +- Exchange (10)
                     :     +- Project (9)
                     :        +- SortMergeJoin LeftOuter (8)
                     :  

### Query 2.5

In [56]:
res = spark.sql(query_2_5())
res.show()
res.explain(mode="formatted")

+---+----+-----+
|day|hour|trips|
+---+----+-----+
|Fri|  18|29050|
|Fri|  17|28034|
|Fri|  19|26656|
|Fri|  16|25645|
|Fri|  15|25578|
|Fri|  14|24746|
|Fri|  22|24015|
|Fri|  13|22190|
|Fri|  23|22100|
|Fri|  21|21976|
|Fri|  20|21168|
|Fri|  12|21024|
|Fri|  11|19852|
|Fri|  10|19559|
|Fri|   9|18192|
|Fri|   8|17323|
|Fri|   7|13043|
|Fri|   0| 8804|
|Fri|   6| 6283|
|Fri|   1| 4805|
+---+----+-----+
only showing top 20 rows
== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Project (2)
                  +- Scan parquet spark_catalog.default.taxi_trips (1)


(1) Scan parquet spark_catalog.default.taxi_trips
Output [1]: [pu_datetime#15599]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/mac/Documents/dev/ID2221/dic/spark_project/spark-warehouse/taxi_trips]
ReadSchema: struct<pu_datetime:timestamp>

(2) Project
Output [2]: [hour(pu_datetime#15599, S

### Query 2.6

In [57]:
res = spark.sql(query_2_6())
res.show()
res.explain(mode="formatted")

+-----+-------+
|month|  trips|
+-----+-------+
|  Dec|     12|
|  Feb|      3|
|  Jan|2964553|
+-----+-------+

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Exchange (4)
            +- HashAggregate (3)
               +- Project (2)
                  +- Scan parquet spark_catalog.default.taxi_trips (1)


(1) Scan parquet spark_catalog.default.taxi_trips
Output [1]: [pu_datetime#15781]
Batched: true
Location: PreparedDeltaFileIndex [file:/Users/mac/Documents/dev/ID2221/dic/spark_project/spark-warehouse/taxi_trips]
ReadSchema: struct<pu_datetime:timestamp>

(2) Project
Output [1]: [date_format(pu_datetime#15781, MMM, Some(Europe/Stockholm)) AS _groupingexpression#15877]
Input [1]: [pu_datetime#15781]

(3) HashAggregate
Input [1]: [_groupingexpression#15877]
Keys [1]: [_groupingexpression#15877]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#15870L]
Results [2]: [_groupingexpression#15877, count#15871L